# Comparing Perturbation Types

This notebook compares the four perturbation kernels available in `ecl.perturbations` and shows how the choice of perturbation affects ECL estimates.

- **Random substitution** ($\Pi^{\text{sub}}$): each perturbed nucleotide is replaced with a uniformly random alternative.
- **Dinucleotide shuffle** ($\Pi^{\text{shuf}}$): preserves dinucleotide frequencies.
- **k-mer Markov** ($\Pi^{\text{Markov}}$): resamples from a Markov model fitted to local context.
- **Generative infilling** ($\Pi^{\text{gen}}$): placeholder using Markov proxy.

Proposition 6.5 in the paper predicts that stronger perturbations yield larger influence values.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from ecl.models.base import SyntheticModel
from ecl.influence import compute_influence_profile
from ecl.ecl import ECL, AECP
from ecl.perturbations import (
    RandomSubstitution,
    DinucleotideShuffle,
    KmerMarkov,
    GenerativeInfilling,
)

## Setup

Create a synthetic model and generate sequences. We use a moderate decay length of 80 bp.

In [ ]:
model = SyntheticModel(seq_length=500, embed_dim=64, decay_length=80.0)

rng = np.random.default_rng(123)
sequences = rng.integers(0, 4, size=(20, 500)).astype(np.int8)
reference = 250

perturbations = {
    "Substitution": RandomSubstitution(),
    "Dinuc. Shuffle": DinucleotideShuffle(),
    "3-mer Markov": KmerMarkov(k=3),
    "Generative": GenerativeInfilling(k=5),
}

print(f"Model: decay_length=80 bp, {model.embedding_dim}-dim embedding")
print(f"Perturbation types: {list(perturbations.keys())}")

## Compute Influence Profiles Under Each Perturbation

In [ ]:
profiles = {}
for name, kernel in perturbations.items():
    distances, influence = compute_influence_profile(
        model_fn=model,
        sequences=sequences,
        reference=reference,
        max_distance=200,
        positions_per_distance=3,
        perturbation=kernel,
        rng=rng,
        show_progress=False,
    )
    profiles[name] = (distances, influence)
    ecl_90 = ECL(distances, influence, beta=0.9)
    aecp = AECP(distances, influence)
    print(f"{name:20s}  ECL_0.9 = {ecl_90:4d} bp   AECP = {aecp:6.1f} bp")

## Visualize: Influence Profiles and ECL Comparison

In [ ]:
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: influence profiles (log scale)
ax = axes[0]
for (name, (d, infl)), color in zip(profiles.items(), colors):
    ax.semilogy(d[1:], infl[1:], label=name, color=color, linewidth=1.5)
ax.set_xlabel("Distance from reference (bp)")
ax.set_ylabel("Influence energy I(d; r)")
ax.set_title("Influence Profiles by Perturbation Type")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: ECL at multiple beta values
ax = axes[1]
betas = [0.5, 0.7, 0.8, 0.9, 0.95]
x_pos = np.arange(len(betas))
width = 0.18
for idx, (name, (d, infl)) in enumerate(profiles.items()):
    ecl_vals = [ECL(d, infl, beta=b) for b in betas]
    ax.bar(x_pos + idx * width, ecl_vals, width, label=name, color=colors[idx], alpha=0.8)
ax.set_xticks(x_pos + 1.5 * width)
ax.set_xticklabels([f"{b:.2f}" for b in betas])
ax.set_xlabel("Beta threshold")
ax.set_ylabel("ECL_beta (bp)")
ax.set_title("ECL Estimates by Perturbation Type")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Discussion

**Key observations:**
- Random substitution produces the largest influence values because it guarantees the perturbed nucleotide differs from the original.
- Dinucleotide shuffle is more conservative, preserving local sequence composition.
- Markov-based perturbations produce the smallest effects since the resampled sequence is statistically similar to the original.
- All perturbation types recover the same qualitative decay shape, confirming the robustness of the ECL framework (Proposition 6.5).

For model comparison, it is important to use the **same perturbation kernel** across models to ensure a fair comparison.